In [1]:
import numpy as np
import pandas as pd
import time

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
path = "movie_data/"

In [4]:
df_items = pd.read_csv(path + "movies.csv")
df_interactions = pd.read_csv(path + "ratings.csv")

In [5]:
df_items.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        if isinstance(x, str):
            # Xóa khoảng trắng giữa các tên riêng (ví dụ: 'Action, Sci-Fi' -> 'action,sci-fi')
            # Sau đó xóa dấu phẩy để tạo thành chuỗi các từ đơn
            return x.replace(" ", "").lower().replace("|", " ")
        else:
            return ''

In [7]:
def create_soup(data_row):
    # Làm sạch các trường dữ liệu
    genres = clean_data(data_row['genres'])
    
    return f"{genres} "

In [8]:
df_items['soup'] = df_items.apply(create_soup, axis=1)
df_items.head()

,movieId,title,genres,soup
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,adventure animation children comedy fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,adventure children fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,comedy romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,comedy drama romance
4,5,Father of the Bride Part II (1995),Comedy,comedy


In [9]:
vectorize = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorize.fit_transform(df_items['soup'])

print(tfidf_matrix.shape)

(9742, 22)


In [10]:
indices = pd.Series(df_items.index, index=df_items['movieId']).drop_duplicates()

In [11]:
def build_user_profile(user_id, interactions_df, latent_matrix, indices):
    user_interactions = interactions_df[interactions_df['userId'] == user_id]
    user_profile = np.zeros(latent_matrix.shape[1], dtype=float)

    for _, row in user_interactions.iterrows():
        item_id = row['movieId']
        rating = row['rating']

        if item_id in indices:
            item_index = indices[item_id]
            item_vector = latent_matrix[item_index].toarray().ravel()
            user_profile += rating * item_vector

    return user_profile

In [12]:
def recommendations_for_user(user_id, interactions_df, latent_matrix, indices, items_df, top_n=10):
    user_profile = np.asarray(
        build_user_profile(user_id, interactions_df, latent_matrix, indices)
    ).ravel()

    similarities = cosine_similarity(user_profile.reshape(1, -1), latent_matrix).ravel()
    top_indices = similarities.argsort()[-top_n:][::-1]
    recommendations = items_df.iloc[top_indices][['movieId', 'title', 'genres']]
    return recommendations

In [13]:
user_profile = build_user_profile(10, df_interactions, tfidf_matrix, indices)
print(user_profile)

[ 40.0808666   54.71065064  30.51696079  25.60734276 139.51106184
  21.91286895   0.         114.98701256  31.98360782   4.21276353
   0.           1.84485218  41.53247469  18.3954875    3.582857
   0.           0.         183.76521246   4.21276353  18.66114265
  10.85216703   0.        ]


In [14]:
rcm = recommendations_for_user(10, df_interactions, tfidf_matrix, indices, df_items)
rcm

,movieId,title,genres
6084,41769,Mozart and the Whale (2005),Comedy|Drama|Romance
1161,1537,Shall We Dance? (Shall We Dansu?) (1996),Comedy|Drama|Romance
6638,56367,Juno (2007),Comedy|Drama|Romance
5535,26599,"Law of Desire (Ley del deseo, La) (1987)",Comedy|Drama|Romance
6309,48342,Conversations with Other Women (2005),Comedy|Drama|Romance
5906,33649,Saving Face (2004),Comedy|Drama|Romance
2448,3261,Singles (1992),Comedy|Drama|Romance
6968,66203,He's Just Not That Into You (2009),Comedy|Drama|Romance
7778,91666,Last Holiday (1950),Comedy|Drama|Romance
6569,55112,Shanghai Kiss (2007),Comedy|Drama|Romance


In [15]:
from sklearn.model_selection import train_test_split

user_counts = df_interactions['userId'].value_counts()
heavy_users = user_counts[user_counts > 50].index
df_heavy = df_interactions[df_interactions['userId'].isin(heavy_users)].copy()

# 2. Áp dụng Scenario 2: Chỉ lấy những phim User thực sự thích (Rating >= 4.0)
df_high_heavy = df_heavy[df_heavy['rating'] >= 4.0].copy()

train_heavy, test_heavy = train_test_split(
    df_high_heavy, test_size=0.2, random_state=42
)

print(f"Số lượng tương tác chất lượng (Train): {len(train_heavy)}")
print(f"Số lượng tương tác chất lượng (Test): {len(test_heavy)}")

Số lượng tương tác chất lượng (Train): 35361
Số lượng tương tác chất lượng (Test): 8841


In [18]:
def evaluate_scenario_2(train_df, test_df, latent_matrix, indices, items_df, k=15):
    precisions = []
    recalls = []
    
    common_users = set(train_df['userId'].unique()) & set(test_df['userId'].unique())
    
    for user_id in common_users:
        user_profile = build_user_profile(user_id, train_df, latent_matrix, indices)
        if np.all(user_profile == 0): continue
        
        user_profile_reshaped = user_profile.reshape(1, -1)
        similarities = cosine_similarity(user_profile_reshaped, latent_matrix).ravel()
        
        user_train_movies = set(train_df[train_df['userId'] == user_id]['movieId'])
        sorted_indices = similarities.argsort()[::-1]
        
        recommended_ids = []
        for idx in sorted_indices:
            m_id = items_df.iloc[idx]['movieId']
            if m_id not in user_train_movies:
                recommended_ids.append(m_id)
            if len(recommended_ids) == k:
                break
        
        actual_viewed_test = set(test_df[test_df['userId'] == user_id]['movieId'])
        
        tp = len(set(recommended_ids) & actual_viewed_test)
        
        precisions.append(tp / k)
        
        recalls.append(tp / len(actual_viewed_test))
        
    # Trung bình cộng toàn hệ thống
    avg_p = np.mean(precisions) if precisions else 0
    avg_r = np.mean(recalls) if recalls else 0
    f1 = (2 * avg_p * avg_r) / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0
    
    return avg_p, avg_r, f1

# Thực thi đánh giá
p_s2, r_s2, f1_s2 = evaluate_scenario_2(train_heavy, test_heavy, tfidf_matrix, indices, df_items, k=15)

print(f"--- KẾT QUẢ THEO SCENARIO 2 (USER PREFERENCES) ---")
print(f"Precision@15: {p_s2:.4f}")
print(f"Recall@15:    {r_s2:.4f}")
print(f"F1 Score:     {f1_s2:.4f}")

--- KẾT QUẢ THEO SCENARIO 2 (USER PREFERENCES) ---
Precision@15: 0.0074
Recall@15:    0.0059
F1 Score:     0.0066
